<a href="https://colab.research.google.com/github/Kanvara001/AI-Prototype2025/blob/main/LabB_DL_shock_edge_workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab B — Edge Computing & Deep Learning at the Bedside

https://kku.world/icu-demo

## ตอนที่ 1: จากข้อมูล → โมเดล → อุปกรณ์ขนาดเล็ก
\
**วิธีใช้:** กดปุ่ม ▶ (Run) ที่มุมซ้ายของแต่ละกล่องทีละกล่อง จากบนลงล่าง — ไม่ต้องแก้โค้ดใดๆ

> ข้อมูลในแล็บนี้เป็น **ข้อมูลจำลอง (synthetic)** ไม่ใช่ข้อมูลผู้ป่วยจริง

---
## โครงสร้างข้อมูลของแล็บนี้

| สิ่งที่กำหนด | ค่า |
|---|---|
| ผู้ป่วย 1 ราย | 1 ตัวอย่าง (sample) |
| อินพุตของ 1 ตัวอย่าง | 720 นาที = 12 ชั่วโมง |
| ตัวแปร (features) | 5 ตัว: HR, MAP, RR, SpO2, Temp |
| หน้าต่างย่อย | 15 นาที **ไม่ทับซ้อนกัน** → 720 / 15 = **48 หน้าต่าง** |
| รูปร่างอินพุตของโมเดล | **(48, 15, 5)** = (หน้าต่าง, นาทีในหน้าต่าง, ตัวแปร) |
| ป้ายกำกับ (Outcome) | 1 = เกิด shock ภายใน **1 ชั่วโมงถัดจากปลายอินพุต**, 0 = ไม่เกิด |

```
   ←──────────── อินพุต 12 ชั่วโมง (นาทีที่ 0-719) ────────────→ | ← 1 ชม. → |
   ├────┬────┬────┬── … 48 หน้าต่าง ละ 15 นาที … ──┬────┬────┤===========▼T0
   0   15   30   45                                      720        780
                                                          ↑ ช่วงนี้โมเดลไม่เห็น    
```


### STEP 0 — เตรียมเครื่องมือ / Set up

In [ ]:
# นำเข้าไลบรารีที่ต้องใช้ทั้งหมด
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import os

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix

# ตั้งค่าการแสดงผล
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (11, 3.2)

# ลองโหลด TensorFlow ถ้าไม่มีก็ยังทำแล็บส่วนอื่นต่อได้
try:
    import tensorflow as tf
    TF = True
except Exception:
    TF = False

print("พร้อมแล้ว / Ready.  TensorFlow available:", TF)

### STEP 1 — สร้างชุดข้อมูล ICU จำลอง / Create small ICU dataset

ผู้ป่วยแต่ละราย: บันทึกสัญญาณชีพทุก **1 นาที** ยาว **780 นาที** = อินพุต 720 นาที + ช่วงทำนาย 60 นาที
ผู้ป่วยที่จะเกิด shock (30%) จะค่อย ๆ ทรุดลงตลอด 12 ชั่วโมง แล้วเกิด shock ในชั่วโมงสุดท้าย

**หมายเหตุ:** สัญญาณรบกวนในเวอร์ชันนี้ทำแบบ "ต่อเนื่อง" (autocorrelated) เหมือนสัญญาณชีพจริง
(ไม่ใช่สุ่มอิสระทีละนาที) และความรุนแรงของแต่ละคนก็สุ่มไม่เท่ากัน → บางรายมีสัญญาณเตือนจาง ๆ
ทำให้การทำนายมีความไม่แน่นอนแบบสมจริง ไม่ง่ายเกินไป

In [ ]:
# --- ค่าคงที่หลักของแล็บ -------------------------------------------------------
RNG = np.random.default_rng(42)

# จำนวนผู้ป่วย ถ้าอยากได้เร็วมาก เปลี่ยนให้น้อย ถ้าอยากให้เสถียรเพิ่มจำนวน
PATIENTS = 80

# สัดส่วนผู้ป่วยที่เกิด event (shock)
EVENT_RATE = 0.3

# ความยาวอินพุต 720 นาที = 12 ชั่วโมง
INPUT_MIN = 720

# ช่วงทำนายล่วงหน้า 60 นาที = 1 ชั่วโมง (โมเดลไม่เห็นช่วงนี้)
GAP_MIN = 60

# ความยาวข้อมูลทั้งหมดต่อผู้ป่วย 1 ราย
TOTAL_MIN = INPUT_MIN + GAP_MIN

# ตัวแปรที่ใช้ 5 ตัว
FEATURES = ["HR", "MAP", "RR", "SpO2", "Temp"]

# ค่าเฉลี่ยปกติของแต่ละสัญญาณ (baseline)
BASE = {"HR": 88, "MAP": 85, "RR": 18, "SpO2": 97, "Temp": 37.0}

# ความแปรปรวนตามธรรมชาติของแต่ละสัญญาณ (noise)
SD = {"HR": 7, "MAP": 6, "RR": 2.2, "SpO2": 1.3, "Temp": 0.28}

# ทิศทางและขนาดการเปลี่ยนแปลงเต็มที่ ณ จุดเกิด shock (หัวใจเร็วขึ้น ความดันตก หายใจเร็ว ออกซิเจนลด ไข้ขึ้น)
DRIFT = {"HR": +25, "MAP": -25, "RR": +5.8, "SpO2": -4.6, "Temp": +0.7}

print(f"ผู้ป่วย {PATIENTS} ราย | อินพุต {INPUT_MIN} นาที | ช่วงทำนาย {GAP_MIN} นาที | รวม {TOTAL_MIN} นาทีต่อราย")

In [ ]:
# --- ฟังก์ชันสร้างสัญญาณรบกวนแบบ "ต่อเนื่อง" ให้กราฟดูเหมือนสัญญาณจริง --------
def smooth_noise(n, sd, rho=0.985):
    # rho ใกล้ 1 = ค่าที่ติดกันคล้ายกันมาก (สัญญาณเรียบ ไม่กระโดดทีละนาที)
    e = RNG.normal(0, sd * np.sqrt(1 - rho ** 2), n)

    # ใช้ผลรวมถ่วงน้ำหนักแบบสะสม (convolution) แทนการวนลูปทีละจุด เพื่อความเร็ว
    w = rho ** np.arange(n)
    x = np.convolve(e, w)[:n]

    # บวกค่าเริ่มต้นที่จางหายไปตามเวลา
    x = x + RNG.normal(0, sd) * w
    return x

print("สร้างฟังก์ชันเรียบร้อย / function ready")

In [ ]:
# --- สร้างข้อมูลจำลองทีละราย ----------------------------------------------------
frames = []

for pid in range(PATIENTS):

    # สุ่มว่าผู้ป่วยรายนี้จะเกิด shock ในชั่วโมงสุดท้ายหรือไม่ (30% ของผู้ป่วยเป็น event)
    outcome = int(RNG.binomial(1, EVENT_RATE))

    # ผู้ป่วยแต่ละคนทรุดลงไม่เท่ากัน จึงสุ่มความรุนแรงประจำตัว (บางรายทรุดน้อยมาก)
    severity = RNG.uniform(0.5, 1.2)

    n = TOTAL_MIN

    # ผู้ป่วยที่จะ shock ทรุดลง 2 จังหวะ:
    #   (1) slow = ค่อย ๆ เปลี่ยนใน 6 ชั่วโมงสุดท้ายของข้อมูลทั้งหมด
    #   (2) fast = ทรุดเร็วใน 3 ชั่วโมงสุดท้าย (ครอบคลุมทั้งปลายอินพุตและช่วงทำนาย)
    ramp = np.zeros(n)
    if outcome == 1:
        slow = np.zeros(n)
        k1 = min(6 * 60, n - 60)
        slow[-k1:] = np.linspace(0, 1, k1) ** 1.7

        fast = np.zeros(n)
        k2 = min(3 * 60, n - 30)
        fast[-k2:] = np.linspace(0, 1, k2) ** 2

        ramp = 0.3 * slow + 0.8 * fast

    # ตารางของผู้ป่วยรายนี้
    block = {"PatientID": pid, "Minute": np.arange(n), "Outcome": outcome}

    # สร้างค่าของแต่ละสัญญาณชีพ = ค่าฐาน + สัญญาณรบกวนต่อเนื่อง + การเปลี่ยนแปลงก่อน shock
    for v in FEATURES:
        sig = BASE[v] + RNG.normal(0, SD[v]) + smooth_noise(n, SD[v])

        if outcome == 1:
            # แต่ละตัวแปรทรุดไม่เท่ากัน (บางตัวแปรของบางคนแทบไม่เปลี่ยน) เหมือนผู้ป่วยจริง
            sig = sig + DRIFT[v] * severity * RNG.uniform(0.4, 1.3) * ramp

        block[v] = np.round(sig, 2)

    frames.append(pd.DataFrame(block))

df = pd.concat(frames, ignore_index=True)

print("ขนาดตาราง / shape:", df.shape)
print("จำนวนผู้ป่วย:", df.PatientID.nunique())
print("สัดส่วนผู้ป่วยที่เกิด event: %.1f%%" % (100 * df.groupby("PatientID").Outcome.first().mean()))
df

#### ตรวจสอบโครงสร้างข้อมูล
ผู้ป่วย 1 ราย ต้องมี 780 แถว และเลข Minute ต้องเรียง 0, 1, 2, … ไม่ขาดหาย

In [ ]:
# --- ตรวจจำนวนแถวต่อผู้ป่วย ----------------------------------------------------
rows_per_patient = df.groupby("PatientID").size()

print("จำนวนแถวต่อผู้ป่วย: ต่ำสุด", rows_per_patient.min(), "| สูงสุด", rows_per_patient.max())

# ตรวจว่าเลขนาทีเรียงต่อกันทีละ 1 จริงหรือไม่
step_ok = df.groupby("PatientID")["Minute"].diff().dropna().eq(1).all()

if step_ok and rows_per_patient.eq(TOTAL_MIN).all():
    print("✅ ทุกรายมี", TOTAL_MIN, "นาที และเรียงต่อกันทีละ 1 นาทีถูกต้อง")
else:
    print("⚠️ โครงสร้างข้อมูลผิดปกติ ตรวจสอบอีกครั้ง")

# ดูข้อมูลของผู้ป่วย
df

### STEP 2 — ดูข้อมูลผู้ป่วย 1 ราย
แถบ **สีเหลือง** คือ 60 นาทีสุดท้าย = ช่วงที่โมเดล **ไม่เห็น** และเป็นช่วงที่ต้องทำนาย

In [ ]:
PATIENT = 0        # <<< เปลี่ยนได้: 0 ถึง PATIENTS-1

# ดึงข้อมูลของผู้ป่วยรายที่เลือก
g = df[df["PatientID"] == PATIENT]

fig, ax = plt.subplots(2, 1, sharex=True, figsize=(11, 5))

# ชั้นบน: ชีพจร และ อัตราการหายใจ
ax[0].plot(g["Minute"], g["HR"], lw=.8, label="HR")
ax[0].plot(g["Minute"], g["RR"], lw=.8, label="RR")

# ชั้นล่าง: ความดันเฉลี่ย และ ออกซิเจนปลายนิ้ว
ax[1].plot(g["Minute"], g["MAP"], lw=.8, label="MAP")
ax[1].plot(g["Minute"], g["SpO2"], lw=.8, label="SpO2")

for a in ax:
    # ระบายช่วง 60 นาทีสุดท้ายที่โมเดลไม่เห็น
    a.axvspan(INPUT_MIN, TOTAL_MIN, color="gold", alpha=.4)
    a.legend(loc="upper left")
    a.grid(alpha=.3)

ax[1].set_xlabel("Minute since start")
ax[0].set_title(f"PatientID {PATIENT}  |  Outcome = {g.Outcome.iloc[0]}   (yellow = 1-hour prediction window)")
plt.tight_layout()
plt.show()

### STEP 3 — จำลองค่าที่หายไป แล้วเติมกลับ / Simulate & fill missing values
ในโลกจริงสายหลุด เครื่องค้าง หรือพยาบาลถอดสายเพื่อทำหัตถการ → ข้อมูลหาย
เราจำลองให้หาย **10%** แล้วเติมด้วยการลากเส้นตรง (linear interpolation) **แยกทีละผู้ป่วย**

In [ ]:
# --- จำลองค่าที่หายไป 10% ในแต่ละตัวแปร ----------------------------------------
for col in FEATURES:
    # สุ่มตำแหน่งที่จะทำให้ค่าหาย
    mask = np.random.rand(len(df)) < 0.10
    df.loc[mask, col] = np.nan

print("จำนวนค่าที่หายไปในแต่ละตัวแปร:")
print(df.isna().sum())

In [ ]:
# --- เติมค่าที่หายไป โดยคำนวณแยกทีละผู้ป่วย -----------------------------------
# เรียงข้อมูลให้ถูกลำดับก่อนเสมอ มิฉะนั้นการ interpolate จะผิด
df = df.sort_values(["PatientID", "Minute"])

df[FEATURES] = (
    df.groupby("PatientID")[FEATURES]
      .transform(lambda x: x.interpolate(method="linear").ffill().bfill())
)

print("จำนวนค่าที่ยังหายอยู่หลังเติม:", int(df[FEATURES].isna().sum().sum()))

In [ ]:
# --- ดูผลการเติมค่าด้วยตา ------------------------------------------------------
g = df[df["PatientID"] == 0].head(240)

plt.plot(g["Minute"], g["HR"], lw=1)
plt.title("PatientID 0: HR after filling missing values (first 4 hours)")
plt.xlabel("Minute")
plt.grid(alpha=.3)
plt.show()

### STEP 4 — ตัดหน้าต่างแบบไม่ทับซ้อน และสร้างป้ายกำกับ / Windowing & labelling

1. ใช้เฉพาะนาทีที่ **0–719** เป็นอินพุต (นาทีที่ 720–779 คือช่วงทำนาย โมเดลห้ามเห็น)
2. ตัดเป็นหน้าต่างละ **15 นาที** โดยเลื่อนทีละ 15 นาที → **ไม่ทับซ้อนกันเลย** (`step = WINDOW`)
3. ได้ 720 / 15 = **48 หน้าต่าง** ต่อผู้ป่วย 1 ราย
4. ป้ายกำกับ = `Outcome` ของผู้ป่วยรายนั้น (1 = เกิด shock ในชั่วโมงถัดไป)

In [ ]:
WINDOW = 15                       # ความยาวหน้าต่าง 15 นาที
STEP = WINDOW                     # ★ เลื่อนทีละ 15 นาที = เท่ากับความยาวหน้าต่าง → ไม่ทับซ้อน ★
N_WINDOW = INPUT_MIN // WINDOW    # จำนวนหน้าต่างต่อผู้ป่วย = 720 / 15 = 48

print(f"หน้าต่างละ {WINDOW} นาที | เลื่อนทีละ {STEP} นาที | ได้ {N_WINDOW} หน้าต่างต่อผู้ป่วย")

#### 4.1 สาธิตการตัดหน้าต่างของผู้ป่วย 1 ราย

In [ ]:
# --- ดูว่าหน้าต่างแต่ละอันครอบคลุมนาทีไหนบ้าง ---------------------------------
demo = df[df["PatientID"] == 0].sort_values("Minute")

# ใช้เฉพาะช่วงอินพุต 720 นาทีแรก
demo_input = demo[demo["Minute"] < INPUT_MIN]

rows = []
values = demo_input[FEATURES].values

for i in range(0, len(values), STEP):
    rows.append({"window_no": len(rows),
                 "minute_from": i,
                 "minute_to": i + WINDOW - 1,
                 "n_minutes": len(values[i:i + WINDOW])})

demo_table = pd.DataFrame(rows)

print("จำนวนหน้าต่างที่ได้:", len(demo_table))
print("Outcome ของผู้ป่วยรายนี้:", demo["Outcome"].iloc[0])

# แสดง 3 หน้าต่างแรก และ 3 หน้าต่างสุดท้าย
demo_table

In [ ]:
demo_input

In [ ]:
# --- วาดให้เห็นว่าหน้าต่างไม่ทับซ้อนกัน --------------------
plt.figure(figsize=(11, 3.2))
plt.plot(demo_input["Minute"].values, demo_input["HR"].values, lw=1, color="black", label="HR")

# ระบายสีสลับกันทีละหน้าต่าง 15 นาที
for i in range(48):
    plt.axvspan(i * WINDOW, (i + 1) * WINDOW, color=["tab:blue", "tab:orange"][i % 2], alpha=.18)

plt.title("Non-overlapping 15-minute windows")
plt.xlabel("Minute")
plt.legend()
plt.grid(alpha=.3)
plt.show()

#### 4.2 ตัดหน้าต่างให้ครบทุกคน → ได้ก้อนข้อมูล 4 มิติ

In [ ]:
# --- สร้าง X และ y จากผู้ป่วยทุกราย --------------------------------------------
X = []
y = []

for pid, patient in df.groupby("PatientID"):

    # เรียงตามเวลาเสมอ
    patient = patient.sort_values("Minute")

    # ★ ใช้เฉพาะ 720 นาทีแรกเป็นอินพุต ตัดชั่วโมงสุดท้ายทิ้ง (ห้ามให้โมเดลเห็น) ★
    patient_input = patient[patient["Minute"] < INPUT_MIN]

    # ดึงเฉพาะคอลัมน์ตัวแปร ออกมาเป็นอาเรย์ (720, 5)
    values = patient_input[FEATURES].values

    patient_windows = []

    # ตัดทีละ 15 นาที โดยเลื่อนทีละ 15 นาที → ไม่ทับซ้อน
    for i in range(0, len(values), STEP):
        patient_windows.append(values[i:i + WINDOW])

    # ผู้ป่วย 1 ราย → (48, 15, 5)
    patient_windows = np.array(patient_windows)

    X.append(patient_windows)
    y.append(patient["Outcome"].iloc[0])

X = np.array(X, dtype="float32")
y = np.array(y)

print("รูปร่างของ X:", X.shape, "→ (ผู้ป่วย, 48 หน้าต่าง, 15 นาที, 5 ตัวแปร)")
print("รูปร่างของ y:", y.shape)
print("จำนวนผู้ป่วยที่เกิด event:", int(y.sum()), "จาก", len(y), "ราย")

### STEP 5 — แบ่งข้อมูลฝึก / ทดสอบ
แบ่งแบบ **stratify** เพื่อให้สัดส่วนผู้ป่วยที่เกิด event เท่ากันทั้งสองฝั่ง
(ที่นี่ผู้ป่วย 1 ราย = 1 ตัวอย่าง ข้อมูลของคนเดียวกันจึงไม่มีทางกระจายไปทั้งสองฝั่ง)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("สัดส่วน event: ฝึก %.1f%% | ทดสอบ %.1f%%" % (100 * y_train.mean(), 100 * y_test.mean()))

In [ ]:
# --- ปรับสเกลข้อมูล (normalization) ------------------------------------------
# คำนวณค่าเฉลี่ยและส่วนเบี่ยงเบนจาก "ฝั่งฝึกเท่านั้น" เพื่อไม่ให้ข้อมูลทดสอบรั่ว
MU = X_train.mean(axis=(0, 1, 2))
SDV = X_train.std(axis=(0, 1, 2)) + 1e-6

X_train_n = ((X_train - MU) / SDV).astype("float32")
X_test_n = ((X_test - MU) / SDV).astype("float32")

print("ค่าเฉลี่ยที่ใช้ปรับสเกล:", np.round(MU, 2))
print("ส่วนเบี่ยงเบนที่ใช้ :", np.round(SDV, 2))

### STEP 6 — โมเดลที่ 1: Machine Learning แบบคลาสสิก (Random Forest)
Random Forest รับข้อมูล 4 มิติไม่ได้ เราจึงสรุปแต่ละหน้าต่างเป็นตัวเลขไม่กี่ตัวก่อน
(ค่าเฉลี่ย, ส่วนเบี่ยงเบน, ต่ำสุด, สูงสุด ของทั้ง 12 ชม. และค่าเฉลี่ยของ 2 ชั่วโมงสุดท้าย)

In [ ]:
# --- ฟังก์ชันสรุปข้อมูล 4 มิติ ให้เป็นตารางฟีเจอร์ 2 มิติ ----------------------
def summarize(A):
    # A มีรูปร่าง (ผู้ป่วย, 48 หน้าต่าง, 15 นาที, 5 ตัวแปร)
    # ยุบมิติ "หน้าต่าง" กับ "นาที" เข้าด้วยกันก่อน → (ผู้ป่วย, 720, 5)
    # แต่เดิมมี 5 ตัวแปร แต่ละตัวแปรมีค่าสถิติ 6 แบบ
    flat = A.reshape(A.shape[0], -1, A.shape[-1])

    m = flat.mean(1)                   # ค่าเฉลี่ยตลอด 12 ชั่วโมง
    s = flat.std(1)                    # ความผันผวน
    lo = flat.min(1)                   # ค่าต่ำสุด
    hi = flat.max(1)                   # ค่าสูงสุด
    d = flat[:, -1, :] - flat[:, 0, :] # ค่าท้ายลบค่าหัว = แนวโน้มรวม
    recent = flat[:, -120:, :].mean(1) # ค่าเฉลี่ย 120 นาทีสุดท้าย = 2 ชั่วโมงล่าสุด

    return np.concatenate([m, s, lo, hi, d, recent], axis=1)


F_train = summarize(X_train)
F_test = summarize(X_test)

print('X_train shape:', X_train.shape)
print("ขนาดตารางฟีเจอร์ฝั่งฝึก:", F_train.shape)
F_train

In [ ]:
# สร้างและฝึกโมเดล
rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(F_train, y_train)

# ทำนายฝั่งทดสอบ
p_rf = rf.predict_proba(F_test)[:, 1]
pred_rf = (p_rf > 0.5).astype(int)

print("Accuracy (Random Forest) = %.3f" % accuracy_score(y_test, pred_rf))
print("AUROC (Random Forest)    = %.3f" % roc_auc_score(y_test, p_rf))

In [ ]:
# --- ดูว่าโมเดลใช้ตัวแปรใดมากที่สุด ---------------------------------------------
names = []
for stat in ["mean", "sd", "min", "max", "delta", "recent2h"]:
    for v in FEATURES:
        names.append(f"{stat}_{v}")

imp = pd.Series(rf.feature_importances_, index=names).sort_values()
imp = imp[-12:]

imp.plot.barh(title="Top features used by the Random Forest")
plt.tight_layout()
plt.show()

### STEP 7 — โมเดลที่ 2: Deep Learning (TimeDistributed LSTM → LSTM)

โครงสร้างนี้อ่านข้อมูล **2 ชั้น**:
1. `TimeDistributed(LSTM(8))` — อ่าน 15 นาทีภายในแต่ละหน้าต่าง แล้วสรุปหน้าต่างนั้นเป็นเวกเตอร์ 8 ตัว (ทำซ้ำเหมือนกันทั้ง 48 หน้าต่าง)
2. `LSTM(8)` — อ่านลำดับของ 48 หน้าต่างนั้น ว่าแนวโน้มตลอด 12 ชั่วโมงเป็นอย่างไร
3. `Dense(1, sigmoid)` — ให้ค่าความน่าจะเป็นที่จะเกิด shock ในชั่วโมงถัดไป

(งานวิจัยจริงของเราใช้ CNN–LSTM ในหลักการเดียวกัน: ชั้นแรกดูรายละเอียดในหน้าต่าง ชั้นสองดูลำดับของหน้าต่าง)

In [ ]:
if TF:
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import TimeDistributed
    from tensorflow.keras.layers import LSTM
    from tensorflow.keras.layers import Dense
    from tensorflow.keras.layers import Dropout
    from tensorflow.keras.callbacks import EarlyStopping

    tf.random.set_seed(42)

    model = Sequential()

    # ชั้นที่ 1: อ่านภายในแต่ละหน้าต่าง 15 นาที (ทำเหมือนกันทั้ง 48 หน้าต่าง)
    model.add(
        TimeDistributed(
            LSTM(8),
            input_shape=(N_WINDOW, WINDOW, len(FEATURES))
        )
    )

    # ชั้นที่ 2: อ่านลำดับของ 48 หน้าต่าง
    model.add(LSTM(32))

    # ลดการจำข้อมูลฝึกมากเกินไป
    model.add(Dropout(0.2))

    # ชั้นซ่อนขนาดเล็ก
    model.add(Dense(16, activation="relu"))

    # ชั้นสุดท้าย: ความน่าจะเป็น 0-1
    model.add(Dense(1, activation="sigmoid"))

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    model.summary()
else:
    print("ไม่พบ TensorFlow — ให้รันไฟล์นี้ใน Google Colab")

การใช้ Time Distributed Layer

- มองภาพข้อมูลเราเป็นกล่อง: ในโปรเจกต์นี้ ข้อมูลผู้ป่วยถูกแบ่งเป็นลำดับชั้น:
  - ระดับใหญ่: ผู้ป่วย 1 คน มี 48 หน้าต่าง (Windows)
  - ระดับย่อย: ในแต่ละหน้าต่าง มี 15 นาที

- TimeDistributed ทำหน้าที่อะไร? ลองจินตนาการว่าคุณมีเครื่องอ่านสัญญาณชีพเครื่องหนึ่งที่เก่งมากในการดูข้อมูล 15 นาที แล้วสรุปออกมาว่า "ช่วงนี้คนไข้ปกติดีไหม"

ถ้าไม่ใช้ TimeDistributed: เราต้องเอาข้อมูล 15 นาทีของหน้าต่างที่ 1 มาใส่เครื่อง แล้วจดสรุป พอหน้าต่างที่ 2 มา แล้วก็ต้องทำแบบเดิมซ้ำอีก 48 ครั้ง ซึ่งยุ่งยาก
ถ้าใช้ TimeDistributed: มันเหมือนการที่เราก๊อปปี้ เครื่องอ่านเครื่องนั้นไปวางไว้หน้าทุก ๆ หน้าต่างพร้อมกัน (ทั้ง 48 หน้าต่าง) เพื่อให้มันทำงานแบบเดียวกันเป๊ะกับข้อมูลทุกช่วงเวลา แล้วสรุปผลของแต่ละช่วงออกมาให้เราทันที

In [ ]:
if TF:
    # หยุดฝึกอัตโนมัติเมื่อผลบนชุด validation ไม่ดีขึ้นแล้ว และย้อนกลับไปใช้น้ำหนักที่ดีที่สุด
    early_stopping_callback = EarlyStopping(
        patience=10,
        restore_best_weights=True
    )

    history = model.fit(
        X_train_n,
        y_train,
        validation_split=0.2,
        epochs=15, # รันแค่ 15 รอบเพื่อสาธิต
        batch_size=32,
        callbacks=[early_stopping_callback],
        verbose=1
    )

    print("ฝึกเสร็จแล้ว จำนวนรอบที่ใช้จริง:", len(history.history["loss"]))

In [ ]:
# --- ดูเส้นการเรียนรู้ ----------------------------------------------------------
if TF:
    plt.plot(history.history["accuracy"], label="train accuracy")
    plt.plot(history.history["val_accuracy"], label="validation accuracy")
    plt.legend()
    plt.grid(alpha=.3)
    plt.title("Learning curve")
    plt.show()

### STEP 8 — วัดผลโมเดล / Evaluate

In [ ]:
if TF:
    # ค่าความสูญเสียและความแม่นยำบนชุดทดสอบ
    loss, acc = model.evaluate(X_test_n, y_test, verbose=0)
    print(f"Accuracy: {acc:.3f}")

    # ความน่าจะเป็นที่โมเดลทำนายออกมา
    y_prob = model.predict(X_test_n, verbose=0)
    print("ตัวอย่างความน่าจะเป็น 10 รายแรก:")
    print(np.round(y_prob[:10].flatten(), 3))

In [ ]:
if TF:
    # แปลงความน่าจะเป็นเป็นคำตอบ 0/1 ด้วยเกณฑ์ 0.5
    y_pred = (y_prob > 0.5).astype(int)

    print("ทำนาย :", y_pred[:10].flatten())
    print("ของจริง:", y_test[:10])
    print("")
    print("AUROC = %.3f" % roc_auc_score(y_test, y_prob.ravel()))

In [ ]:
if TF:
    # ตารางความสับสน (confusion matrix)
    cm = confusion_matrix(y_test, y_pred)
    print(cm)
    print("")

    # อธิบายความหมายของแต่ละช่องให้ชัด
    tn, fp, fn, tp = cm.ravel()
    print("ทำนายถูกว่าไม่เกิด (TN):", tn)
    print("เตือนผิด         (FP):", fp)
    print("พลาดไม่เตือน     (FN):", fn)
    print("จับได้ถูกต้อง      (TP):", tp)
    print("")
    print("Sensitivity (จับ event ได้กี่ %%): %.1f%%" % (100 * tp / max(1, tp + fn)))
    print("Precision   (เตือนแล้วถูกกี่ %%): %.1f%%" % (100 * tp / max(1, tp + fp)))

### STEP 9 — บันทึกโมเดล และย่อให้ลงอุปกรณ์เล็ก / Save & shrink for the edge
- `.keras` / `.h5` = โมเดลเต็ม สำหรับเซิร์ฟเวอร์
- `.tflite` = โมเดลย่อ สำหรับ Raspberry Pi หรือมือถือ
- `scaler.npy` = ค่าเฉลี่ย/ส่วนเบี่ยงเบนที่ใช้ปรับสเกล **ต้องส่งไปพร้อมโมเดลเสมอ**

In [ ]:
if TF:
    # บันทึกโมเดลรูปแบบใหม่ของ Keras
    model.save("shock_lstm.keras")

    # บันทึกรูปแบบเดิม HDF5
    model.save("shock_lstm.h5")

    # แปลงเป็น TensorFlow Lite พร้อมบีบขนาด
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]

    # LSTM ต้องเปิด SELECT_TF_OPS เพราะ TFLite พื้นฐานไม่รองรับคำสั่งบางตัว
    conv._experimental_lower_tensor_list_ops = False
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]

    tflite_bytes = conv.convert()
    with open("shock_lstm.tflite", "wb") as f:
        f.write(tflite_bytes)

    # บันทึกค่าปรับสเกลไปด้วย มิฉะนั้นเครื่องปลายทางจะคำนวณผิด
    np.save("scaler.npy", np.vstack([MU, SDV]))

    for name in ["shock_lstm.keras", "shock_lstm.h5", "shock_lstm.tflite"]:
        print(f"{name:22s} {os.path.getsize(name)/1024:8.1f} KB")

In [ ]:
# --- บันทึกโมเดล Random Forest ไว้เปรียบเทียบขนาด ------------------------------
import joblib

joblib.dump(rf, "shock_rf.joblib")
print("shock_rf.joblib      %8.1f KB  (Random Forest)" % (os.path.getsize("shock_rf.joblib") / 1024))

### STEP 10 — จำลองการทำงานบนอุปกรณ์ข้างเตียง / Edge simulation
โหลดโมเดล `.tflite` ขึ้นมา แล้วป้อนข้อมูลผู้ป่วยทีละราย เหมือนที่เครื่องข้างเตียงจะทำ
วัด **เวลาที่ใช้ทำนาย 1 ครั้ง** และตรวจว่าคำตอบตรงกับโมเดลเต็มหรือไม่

In [ ]:
# --- เตรียมตัวทำนายแบบ edge ----------------------------------------------------
if TF:
    try:
        # โหลดโมเดล TFLite (บางเครื่องต้องมี Flex delegate สำหรับ LSTM)
        itp = tf.lite.Interpreter(model_path="shock_lstm.tflite")
        itp.allocate_tensors()
        USE_TFLITE = True
        print("ใช้โมเดล .tflite สำหรับจำลอง edge")
    except Exception as e:
        # ถ้าโหลดไม่ได้ ให้ถอยไปใช้โมเดล Keras แทน (ผลลัพธ์เท่ากัน)
        print("โหลด .tflite ไม่สำเร็จ (%s) → ใช้โมเดล Keras แทน" % type(e).__name__)
        itp = tf.keras.models.load_model("shock_lstm.keras")
        USE_TFLITE = False

    def predict_edge(one_patient):
        # one_patient มีรูปร่าง (48, 15, 5) และถูกปรับสเกลมาแล้ว
        z = one_patient[None].astype("float32")

        if USE_TFLITE:
            inp_details = itp.get_input_details()[0]
            out_details = itp.get_output_details()[0]
            itp.set_tensor(inp_details["index"], z)
            itp.invoke()
            return float(itp.get_tensor(out_details["index"])[0, 0])
        else:
            return float(itp.predict(z, verbose=0)[0, 0])

    print("ฟังก์ชันทำนายบน edge พร้อมใช้งาน")

In [ ]:
# --- ป้อนผู้ป่วยทีละรายเหมือนเครื่องข้างเตียง ----------------------------------
if TF:
    edge_probs = []
    latencies = []

    for i in range(len(X_test_n)):
        # จับเวลาเริ่ม
        t0 = time.perf_counter()

        # ทำนาย 1 ราย
        p = predict_edge(X_test_n[i])

        # จับเวลาที่ใช้ แปลงเป็นมิลลิวินาที
        latencies.append((time.perf_counter() - t0) * 1000)
        edge_probs.append(p)

    edge_probs = np.array(edge_probs)
    edge_pred = (edge_probs > 0.5).astype(int)

    print("จำนวนผู้ป่วยที่ทำนาย:", len(edge_pred))
    print("เวลาทำนายเฉลี่ยต่อ 1 ราย: %.2f ms" % np.mean(latencies))
    print("Accuracy บนอุปกรณ์ edge : %.3f" % accuracy_score(y_test, edge_pred))
    print("ตรงกับโมเดลเต็มกี่ราย    : %d/%d" % (int((edge_pred.ravel() == y_pred.ravel()).sum()), len(y_test)))

In [ ]:
# --- ดูคะแนนความเสี่ยงของผู้ป่วยแต่ละรายในชุดทดสอบ ------------------------------
if TF:
    order = np.argsort(edge_probs)

    colors = ["tab:green" if v == 0 else "tab:red" for v in y_test[order]]

    plt.figure(figsize=(11, 3.4))
    plt.bar(range(len(order)), edge_probs[order], color=colors)
    plt.axhline(0.5, ls="--", c="grey")
    plt.ylabel("predicted risk")
    plt.xlabel("test patients (sorted by risk)")
    plt.title("Red = truly had shock, Green = no shock")
    plt.grid(alpha=.3, axis="y")
    plt.show()

### สรุป / Wrap-up

**ข้อมูลดิบทุก 1 นาที → เติมค่าที่หาย → ตัดหน้าต่าง 15 นาทีแบบไม่ทับซ้อน (48 หน้าต่าง) → (48, 15, 5) → TimeDistributed LSTM → .tflite → ทำนายข้างเตียง**

1. **รูปร่างข้อมูลคือหัวใจ** — (ผู้ป่วย, 48 หน้าต่าง, 15 นาที, 5 ตัวแปร) กำหนดสถาปัตยกรรมโมเดลไปในตัว
2. **หน้าต่างไม่ทับซ้อน** ทำให้แต่ละหน้าต่างเป็นข้อมูลอิสระ ไม่นับซ้ำ
3. **เว้นช่วง 1 ชั่วโมงก่อนเกิดเหตุ** เพราะการเตือนต้องมาก่อนล่วงหน้าพอให้ทีมเตรียมตัว และกัน data leakage
4. **ปรับสเกลด้วยค่าจากชุดฝึกเท่านั้น** และต้องส่งไฟล์ scaler ไปกับโมเดลเสมอ
5. **.tflite เล็กและเร็ว** → รันข้างเตียงได้ แม้เน็ตหลุด ข้อมูลผู้ป่วยไม่ต้องออกจากโรงพยาบาล

ไฟล์ที่ได้: `shock_lstm.keras`, `shock_lstm.h5`, `shock_lstm.tflite`, `scaler.npy`, `shock_rf.joblib`